# CSDN技术博客AI领域文章多维数据挖掘分析

## 项目概述
本项目通过网络爬虫从CSDN搜索API采集5821篇AI领域技术博客文章数据，采用多种数据挖掘方法进行全面分析。

### 分析方法
1. 描述性统计分析
2. TF-IDF文本特征提取
3. LDA主题建模
4. K-Means聚类分析
5. Apriori关联规则挖掘
6. 时间序列趋势分析
7. 多模型分类预测（LR/RF/XGBoost/GBDT）
8. 技术态度倾向分析（领域词典法）
9. 异常检测（Isolation Forest / LOF）

## 0. 数据采集：CSDN技术博客爬虫

通过CSDN搜索API采集AI领域技术博客文章数据。

**API接口**: `https://so.csdn.net/api/v3/search?q={关键词}&t=blog&p={页码}&s=0&o=1`

**采集关键词**: 机器学习、大模型、深度学习、自然语言处理、数据挖掘、ChatGPT、AIGC、强化学习、计算机视觉、Transformer、卷积神经网络

**每页返回**: 30条记录


In [ ]:
import json, time, re, csv, os
import urllib.request, urllib.parse

# ============ 爬虫配置 ============
KEYWORDS = ['机器学习', '大模型', '深度学习', '自然语言处理', '数据挖掘',
            'ChatGPT', 'AIGC', '强化学习', '计算机视觉', 'Transformer', '卷积神经网络']
PAGES_PER_KEYWORD = 80       # 每个关键词爬80页（每页30条）
DELAY = 0.8                   # 请求间隔（秒）
OUTPUT_FILE = 'csdn_articles_raw.csv'

# ============ 核心函数 ============
def fetch_search_page(keyword, page):
    \"\"\"获取CSDN搜索结果的一页\"\"\"
    params = urllib.parse.urlencode({'q': keyword, 't': 'blog', 'p': page, 's': 0, 'o': 1})
    url = f'https://so.csdn.net/api/v3/search?{params}'
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        'Accept': 'application/json',
        'Referer': 'https://so.csdn.net/',
    }
    req = urllib.request.Request(url, headers=headers)
    resp = urllib.request.urlopen(req, timeout=15)
    data = json.loads(resp.read().decode('utf-8'))
    return data.get('result_vos', [])

def clean_html(text):
    \"\"\"去除HTML标签\"\"\"
    if not text: return ''
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def parse_article(item, keyword):
    \"\"\"从搜索结果条目中提取文章信息\"\"\"
    return {
        'article_id': item.get('articleid', '') or item.get('id', ''),
        'title': clean_html(item.get('title', '')),
        'author': item.get('author', ''),
        'nickname': item.get('nickname', ''),
        'url': item.get('url', '').split('?')[0],
        'publish_date': item.get('create_time_str', ''),
        'views': int(item.get('view', 0) or 0),
        'likes': int(item.get('digg', 0) or 0),
        'comments': int(item.get('comment', 0) or 0),
        'collections': int(item.get('collections', 0) or 0),
        'description': clean_html(item.get('description', '') or item.get('digest', '') or ''),
        'language_tags': item.get('language', '') or '',
        'search_keyword': keyword,
    }

print("爬虫函数定义完成")

In [ ]:
# ============ 主爬取流程（仅供复现，无需再次执行）============
all_articles = {}  # article_id去重
total_fetched = 0
total_errors = 0

for keyword in KEYWORDS:
    print(f"爬取【{keyword}】...")
    consecutive_errors = 0
    for page in range(1, PAGES_PER_KEYWORD + 1):
        try:
            items = fetch_search_page(keyword, page)
            if not items: break
            new_count = 0
            for item in items:
                article = parse_article(item, keyword)
                aid = article['article_id']
                if aid and aid not in all_articles:
                    all_articles[aid] = article
                    new_count += 1
                elif aid and aid in all_articles:
                    existing = all_articles[aid]['search_keyword']
                    if keyword not in existing:
                        all_articles[aid]['search_keyword'] = existing + ',' + keyword
            total_fetched += len(items)
            if page % 20 == 0:
                print(f"  第{page}页: +{new_count}条, 累计去重{len(all_articles)}条")
            consecutive_errors = 0
            time.sleep(DELAY)
        except Exception as e:
            consecutive_errors += 1
            total_errors += 1
            if consecutive_errors >= 5:
                print(f"  连续错误, 停止"); break
            time.sleep(2)

print(f"\n爬取完成: 总获取{total_fetched}条, 去重后{len(all_articles)}条, 错误{total_errors}次")

# 保存CSV
fieldnames = ['article_id','title','author','nickname','url','publish_date',
              'views','likes','comments','collections','description','language_tags','search_keyword']
with open(OUTPUT_FILE, 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_articles.values())
print(f"已保存: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1024/1024:.2f} MB)")

In [ ]:
# ============ 数据清洗 ============
import pandas as pd

df_raw = pd.read_csv(OUTPUT_FILE, encoding='utf-8-sig')
print(f"原始记录: {len(df_raw)}")

cleaned = []
for _, r in df_raw.iterrows():
    title = re.sub(r'<[^>]+>', '', str(r['title'])).strip()
    if len(title) < 5: continue
    date = str(r['publish_date'])
    try:
        year = int(date.split('-')[0])
        if year < 2015 or year > 2026: continue
    except: continue
    tags = str(r.get('language_tags', ''))
    tag_list = [t.strip() for t in tags.split(',') if t.strip() and t.strip() != '灬']
    cleaned.append({
        'article_id': r['article_id'], 'title': title,
        'author': r['author'], 'nickname': r['nickname'], 'url': r['url'],
        'publish_date': date, 'publish_year': year,
        'views': int(r['views'] or 0), 'likes': int(r['likes'] or 0),
        'comments': int(r['comments'] or 0), 'collections': int(r['collections'] or 0),
        'description': str(r['description']).strip(),
        'language_tags': ','.join(tag_list), 'search_keyword': r['search_keyword'],
    })

clean_file = 'csdn_ml_articles_5821.csv'
pd.DataFrame(cleaned).to_csv(clean_file, index=False, encoding='utf-8-sig')
print(f"清洗后: {len(cleaned)}条 -> {clean_file}")

## 1. 环境配置与数据加载

In [ ]:
import sys, os, json, warnings, re
sys.stdout.reconfigure(encoding='utf-8')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
import seaborn as sns
from collections import Counter
import jieba
from wordcloud import WordCloud

# 学术风格配置
FP = FontProperties(fname='C:/Windows/Fonts/simhei.ttf', size=11)
FP_TITLE = FontProperties(fname='C:/Windows/Fonts/simhei.ttf', size=13)
FP_SMALL = FontProperties(fname='C:/Windows/Fonts/simhei.ttf', size=9)
FP_TINY = FontProperties(fname='C:/Windows/Fonts/simhei.ttf', size=7)
FP_LEGEND = FontProperties(fname='C:/Windows/Fonts/simhei.ttf', size=10)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
COLORS = ['#2C5F8A','#D4782F','#4A9B6E','#C44E52','#8172B2','#8C8C8C','#CCB974','#64B5CD']
sns.set_style("whitegrid", {'grid.linestyle':'--','grid.alpha':0.3})
print("环境配置完成")

In [ ]:
# 加载数据
df = pd.read_csv('csdn_ml_articles_5821.csv', encoding='utf-8-sig')
df['publish_date'] = pd.to_datetime(df['publish_date'], errors='coerce')
df['publish_month'] = df['publish_date'].dt.to_period('M')
df['publish_year'] = df['publish_date'].dt.year
def parse_tags(s):
    if pd.isna(s) or not s: return []
    return [t.strip().lower() for t in str(s).split(',') if t.strip() and len(t.strip())>1]
df['tag_list'] = df['language_tags'].apply(parse_tags)
print(f"数据量: {len(df)} 条, {len(df.columns)} 字段")
df.head()

## 2. 描述性统计分析

In [ ]:
print("=== 数值字段统计 ===")
for col in ['views','likes','comments','collections']:
    d = df[col]
    print(f"{col}: mean={d.mean():.1f}, median={d.median():.0f}, std={d.std():.1f}, max={d.max():.0f}")
print("\n=== 热度分布 ===")
for label,lo,hi in [('冷门(<100)',0,100),('普通(100-999)',100,1000),('热门(1000-9999)',1000,10000),('爆款(≥10000)',10000,float('inf'))]:
    cnt = ((df['views']>=lo)&(df['views']<hi)).sum()
    print(f"  {label}: {cnt}篇 ({cnt/len(df)*100:.1f}%)")

In [ ]:
# 图1: 年度发文量分布
fig, ax = plt.subplots(figsize=(10,5))
yc = df['publish_year'].value_counts().sort_index()
bars = ax.bar(yc.index, yc.values, color=COLORS[0], alpha=0.85, edgecolor='white', linewidth=0.5)
for b,v in zip(bars,yc.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+15, str(v), ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xlabel('年份',fontproperties=FP); ax.set_ylabel('文章数量',fontproperties=FP)
ax.set_title('CSDN技术博客年度发文量分布',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.set_xticks(yc.index)
for l in ax.get_xticklabels(): l.set_fontproperties(FP_SMALL)
for l in ax.get_yticklabels(): l.set_fontproperties(FP_SMALL)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图2: 各关键词文章数量
fig, ax = plt.subplots(figsize=(9,5))
kw_counts = Counter()
for tags in df['search_keyword']:
    if pd.notna(tags):
        for kw in str(tags).split(','):
            kw=kw.strip()
            if kw: kw_counts[kw]+=1
kw_df = pd.DataFrame(kw_counts.items(),columns=['关键词','数量']).sort_values('数量',ascending=True)
bars = ax.barh(kw_df['关键词'],kw_df['数量'],color=[COLORS[i%len(COLORS)] for i in range(len(kw_df))],alpha=0.85,edgecolor='white')
for b,v in zip(bars,kw_df['数量']):
    ax.text(b.get_width()+5,b.get_y()+b.get_height()/2,str(v),va='center',fontsize=9)
ax.set_xlabel('文章数量',fontproperties=FP)
ax.set_title('各搜索关键词对应的文章数量',fontproperties=FP_TITLE,weight='bold',pad=12)
for l in ax.get_yticklabels(): l.set_fontproperties(FP)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图3: 浏览量/点赞/评论分布 (截断P99)
fig, axes = plt.subplots(1,3,figsize=(15,5))
for ax,(col,label,color) in zip(axes,[('views','浏览量',COLORS[0]),('likes','点赞数',COLORS[1]),('comments','评论数',COLORS[2])]):
    data=df[col]; p99=data.quantile(0.99); clipped=data[data<=p99]
    ax.hist(clipped,bins=50,color=color,alpha=0.8,edgecolor='white',linewidth=0.3)
    med=data.median(); ax.axvline(med,color='red',linestyle='--',linewidth=1.5,label=f'中位数={med:.0f}')
    ax.set_xlabel(label,fontproperties=FP); ax.set_ylabel('文章数量',fontproperties=FP)
    ax.set_title(f'{label}分布',fontproperties=FP_TITLE,weight='bold')
    for l in ax.get_xticklabels(): l.set_fontproperties(FP_TINY)
    for l in ax.get_yticklabels(): l.set_fontproperties(FP_SMALL)
    ax.legend(prop=FP_LEGEND,fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 3. TF-IDF文本特征提取与NLP分析

In [ ]:
stopwords = set(['的','了','是','在','和','与','为','中','对','等','通过','进行','可以','使用','一个','以及',
    '我们','并','将','但','也','不','有','上','下','其','这','那','一','本文','方法','模型','数据','算法',
    '基于','实现','技术','分析','学习','应用','利用','主要','介绍','详细','需要','包括','相关','结合','提出',
    '如何','什么','为什么','怎么','这个','那个','已经','如果','虽然','但是','因为','所以','而且','或者','可能',
    '应该','非常','比较','之间','之后','之前','以上','以下','其中','关于','对于','从','到','用','做','看','说',
    '让','把','被','给','向','跟','比','又','还','再','就','都','只','最','更','很','太','没','着','过','地',
    '得','呢','吗','吧','啊','哦','嗯','一种','该','它们','然后','同时','此外','提供','支持','探讨','内容',
    '涵盖','文章','领域'])
all_words = []
for desc in df['description']:
    if pd.notna(desc) and len(str(desc))>5:
        words = jieba.lcut(str(desc))
        words = [w.strip() for w in words if len(w.strip())>=2 and w.strip() not in stopwords and not w.strip().isdigit()]
        all_words.extend(words)
word_freq = Counter(all_words)
print(f"分词完成: 总词数={len(all_words)}, 去重={len(word_freq)}")
print(f"Top 15: {word_freq.most_common(15)}")

In [ ]:
# 图4: Top30高频词
fig, ax = plt.subplots(figsize=(10,7))
top30 = pd.DataFrame(word_freq.most_common(30),columns=['词语','频次']).sort_values('频次',ascending=True)
bars = ax.barh(top30['词语'],top30['频次'],color=COLORS[0],alpha=0.85,edgecolor='white')
for b,v in zip(bars,top30['频次']):
    ax.text(b.get_width()+3,b.get_y()+b.get_height()/2,str(v),va='center',fontsize=9)
ax.set_xlabel('频次',fontproperties=FP)
ax.set_title('CSDN技术博客摘要Top30高频词',fontproperties=FP_TITLE,weight='bold',pad=12)
for l in ax.get_yticklabels(): l.set_fontproperties(FP)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图5: 词云
fig, ax = plt.subplots(figsize=(10,6))
wc = WordCloud(width=1200,height=600,background_color='white',font_path='C:/Windows/Fonts/simhei.ttf',
               max_words=150,colormap='Blues',prefer_horizontal=0.7,min_font_size=8)
wc.generate_from_frequencies(dict(word_freq.most_common(150)))
ax.imshow(wc,interpolation='bilinear'); ax.axis('off')
ax.set_title('CSDN技术博客摘要词云',fontproperties=FP_TITLE,weight='bold',pad=12)
plt.tight_layout(); plt.show()

In [ ]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=500,tokenizer=jieba.lcut,stop_words=list(stopwords),min_df=5,max_df=0.8,ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(df['description'].fillna('').astype(str).tolist())
print(f"TF-IDF矩阵: {tfidf_matrix.shape}")

## 4. LDA主题建模

In [ ]:
from gensim import corpora, models as gmodels
texts = []
for desc in df['description']:
    if pd.notna(desc) and len(str(desc))>5:
        words = jieba.lcut(str(desc))
        words = [w.strip() for w in words if len(w.strip())>=2 and w.strip() not in stopwords and not w.strip().isdigit()]
        texts.append(words)
dictionary = corpora.Dictionary(texts)
dictionary.filter_extremes(no_below=5,no_above=0.8)
corpus = [dictionary.doc2bow(t) for t in texts]
coherence_scores = []; K_range = range(3,11)
for k in K_range:
    lda = gmodels.LdaModel(corpus,num_topics=k,id2word=dictionary,passes=15,random_state=42)
    cm = gmodels.CoherenceModel(model=lda,texts=texts,dictionary=dictionary,coherence='u_mass')
    coherence_scores.append(cm.get_coherence())
    print(f"K={k}, coherence={cm.get_coherence():.4f}")
best_k = list(K_range)[np.argmax(coherence_scores)]
print(f"\n最优K={best_k}")

In [ ]:
lda_final = gmodels.LdaModel(corpus,num_topics=best_k,id2word=dictionary,passes=20,random_state=42)
# 图6: 一致性评估
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(list(K_range),coherence_scores,'o-',color=COLORS[0],linewidth=2,markersize=8)
ax.axvline(best_k,color=COLORS[3],linestyle='--',linewidth=1.5,label=f'最优K={best_k}')
ax.set_xlabel('主题数K',fontproperties=FP); ax.set_ylabel('一致性得分(u_mass)',fontproperties=FP)
ax.set_title('LDA主题模型一致性评估',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.legend(prop=FP_LEGEND)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
for i in range(best_k):
    print(f"Topic {i+1}: {[(w,round(p,4)) for w,p in lda_final.show_topic(i,topn=8)]}")

In [ ]:
# 图7: LDA热力图
topic_words = {}
for i in range(best_k):
    topic_words[f'Topic {i+1}'] = [(w,round(p,4)) for w,p in lda_final.show_topic(i,topn=8)]
all_tw = set()
for tw in topic_words.values():
    for w,p in tw: all_tw.add(w)
all_tw_list = sorted(all_tw)[:25]
hm = np.zeros((best_k,len(all_tw_list)))
for i in range(best_k):
    d = dict(topic_words[f'Topic {i+1}'])
    for j,w in enumerate(all_tw_list): hm[i,j]=d.get(w,0)
fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(hm,xticklabels=all_tw_list,yticklabels=[f'主题{i+1}' for i in range(best_k)],
            cmap='YlOrRd',annot=True,fmt='.3f',linewidths=0.5,ax=ax)
ax.set_title(f'LDA主题-关键词分布热力图(K={best_k})',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.set_xlabel('关键词',fontproperties=FP); ax.set_ylabel('主题',fontproperties=FP)
plt.xticks(rotation=45,ha='right')
for l in ax.get_xticklabels(): l.set_fontproperties(FP_SMALL)
for l in ax.get_yticklabels(): l.set_fontproperties(FP)
plt.tight_layout(); plt.show()

## 5. K-Means聚类分析

In [ ]:
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
svd = TruncatedSVD(n_components=50,random_state=42)
tfidf_reduced = svd.fit_transform(tfidf_matrix)
print(f"SVD: {tfidf_matrix.shape}->{tfidf_reduced.shape}, 解释方差={svd.explained_variance_ratio_.sum():.3f}")
inertias=[]; sils=[]; K_km=range(2,11)
for k in K_km:
    km=MiniBatchKMeans(n_clusters=k,random_state=42,n_init=5,batch_size=1000); km.fit(tfidf_reduced)
    inertias.append(km.inertia_); sils.append(silhouette_score(tfidf_reduced,km.labels_,sample_size=3000))
    print(f"K={k}, Inertia={km.inertia_:.1f}, Sil={sils[-1]:.4f}")
best_k_km = list(K_km)[np.argmax(sils)]
print(f"\n最优K={best_k_km}")

In [ ]:
km_final = MiniBatchKMeans(n_clusters=best_k_km,random_state=42,n_init=10,batch_size=1000)
cluster_labels = km_final.fit_predict(tfidf_reduced)
# 图8: 肘部法+轮廓系数
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))
ax1.plot(list(K_km),inertias,'o-',color=COLORS[0],linewidth=2,markersize=8)
ax1.set_xlabel('聚类数K',fontproperties=FP); ax1.set_ylabel('SSE',fontproperties=FP)
ax1.set_title('K-Means肘部法则',fontproperties=FP_TITLE,weight='bold',pad=12)
ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
ax2.plot(list(K_km),sils,'s-',color=COLORS[1],linewidth=2,markersize=8)
ax2.axvline(best_k_km,color=COLORS[3],linestyle='--',linewidth=1.5,label=f'最优K={best_k_km}')
ax2.set_xlabel('聚类数K',fontproperties=FP); ax2.set_ylabel('轮廓系数',fontproperties=FP)
ax2.set_title('K-Means轮廓系数评估',fontproperties=FP_TITLE,weight='bold',pad=12)
ax2.legend(prop=FP_LEGEND)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图9: PCA可视化
pca = PCA(n_components=2,random_state=42)
pca_2d = pca.fit_transform(tfidf_reduced)
fig, ax = plt.subplots(figsize=(10,8))
for i in range(best_k_km):
    mask=cluster_labels==i
    ax.scatter(pca_2d[mask,0],pca_2d[mask,1],c=COLORS[i%len(COLORS)],alpha=0.4,s=15,label=f'簇{i+1}(n={mask.sum()})')
ax.set_xlabel(f'PC1({pca.explained_variance_ratio_[0]*100:.1f}%)',fontproperties=FP)
ax.set_ylabel(f'PC2({pca.explained_variance_ratio_[1]*100:.1f}%)',fontproperties=FP)
ax.set_title(f'K-Means聚类PCA可视化(K={best_k_km})',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.legend(prop=FP_SMALL,loc='upper right',ncol=2)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Apriori关联规则挖掘

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules as ar_func
all_tags_flat = []
for tags in df['tag_list']: all_tags_flat.extend(tags)
tag_freq_all = Counter(all_tags_flat)
top_tags = [t for t,c in tag_freq_all.most_common(30)]
one_hot = [{tag:(tag in tags) for tag in top_tags} for tags in df['tag_list']]
tag_df = pd.DataFrame(one_hot).fillna(False).astype(bool)
frequent_items = apriori(tag_df,min_support=0.02,use_colnames=True)
print(f"频繁项集: {len(frequent_items)}")
rules = pd.DataFrame()
if len(frequent_items)>0:
    rules = ar_func(frequent_items,metric='confidence',min_threshold=0.3,num_itemsets=len(frequent_items))
    rules = rules.sort_values('lift',ascending=False)
    print(f"关联规则: {len(rules)}")
    for _,r in rules.head(10).iterrows():
        print(f"  {', '.join(list(r['antecedents']))} -> {', '.join(list(r['consequents']))}  lift={r['lift']:.2f}")

In [ ]:
# 图10: 关联规则气泡图
if len(rules)>0:
    fig,ax=plt.subplots(figsize=(12,8))
    tr=rules.head(20)
    sc=ax.scatter(tr['support'],tr['confidence'],s=tr['lift']*100,c=tr['lift'],cmap='RdYlBu_r',alpha=0.7,edgecolors='gray',linewidths=0.5)
    for _,r in tr.iterrows():
        ax.annotate(f"{', '.join(list(r['antecedents']))}→{', '.join(list(r['consequents']))}",(r['support'],r['confidence']),fontsize=6,alpha=0.8)
    ax.set_xlabel('支持度',fontproperties=FP); ax.set_ylabel('置信度',fontproperties=FP)
    ax.set_title('技术标签关联规则分布(气泡大小=提升度)',fontproperties=FP_TITLE,weight='bold',pad=12)
    plt.colorbar(sc,label='提升度(Lift)')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout(); plt.show()

In [ ]:
# 图11: Top20标签频次
fig, ax = plt.subplots(figsize=(10,7))
tcdf = pd.DataFrame(tag_freq_all.most_common(20),columns=['标签','频次']).sort_values('频次',ascending=True)
bars = ax.barh(tcdf['标签'],tcdf['频次'],color=COLORS[0],alpha=0.85,edgecolor='white')
for b,v in zip(bars,tcdf['频次']):
    ax.text(b.get_width()+5,b.get_y()+b.get_height()/2,str(v),va='center',fontsize=9)
ax.set_xlabel('出现频次',fontproperties=FP)
ax.set_title('Top20技术标签出现频次',fontproperties=FP_TITLE,weight='bold',pad=12)
for l in ax.get_yticklabels(): l.set_fontproperties(FP)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 7. 时间序列趋势分析

In [ ]:
# 图12: 月度发文趋势
monthly = df.groupby('publish_month').size().reset_index(name='count')
fig, ax = plt.subplots(figsize=(14,5))
ax.plot(range(len(monthly)),monthly['count'].values,'-',color=COLORS[0],linewidth=1.5,alpha=0.8)
ax.fill_between(range(len(monthly)),monthly['count'].values,alpha=0.12,color=COLORS[0])
for i in range(0,len(monthly),6):
    ax.annotate(str(monthly.iloc[i]['publish_month']),(i,monthly.iloc[i]['count']),fontsize=7,rotation=45,ha='right',fontproperties=FP_TINY)
ax.set_xlabel('月份',fontproperties=FP); ax.set_ylabel('文章数量',fontproperties=FP)
ax.set_title('AI领域月度发文量趋势(2015-2026)',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图13: 各关键词月度热度
kw_m=df.explode('search_keyword'); kw_m['search_keyword']=kw_m['search_keyword'].str.strip()
kw_mc=kw_m.groupby(['publish_month','search_keyword']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14,6))
for i,kw in enumerate(['机器学习','大模型','深度学习','自然语言处理','数据挖掘']):
    if kw in kw_mc.columns:
        ax.plot(range(len(kw_mc)),kw_mc[kw].values,'-',color=COLORS[i],linewidth=1.5,label=kw,alpha=0.85)
ax.set_xlabel('月份',fontproperties=FP); ax.set_ylabel('文章数量',fontproperties=FP)
ax.set_title('各技术领域月度热度演变',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.legend(prop=FP_LEGEND,loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 8. 分类模型：文章热度预测

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
X_num=df[['likes','comments','collections']].values
tag_feat=tag_df[top_tags].values
X=np.hstack([X_num,tag_feat,tfidf_matrix.toarray()[:,:100]])
y_bin=(df['views']>=1000).astype(int)
X_tr,X_te,y_tr,y_te=train_test_split(X,y_bin,test_size=0.2,random_state=42,stratify=y_bin)
print(f"训练:{X_tr.shape[0]}, 测试:{X_te.shape[0]}, 正例:{y_bin.mean():.3f}")
md_r={}
for nm,m in [('LR',LogisticRegression(max_iter=1000,random_state=42)),
             ('RF',RandomForestClassifier(n_estimators=200,max_depth=15,random_state=42,n_jobs=-1)),
             ('XGB',xgb.XGBClassifier(n_estimators=200,max_depth=6,learning_rate=0.1,random_state=42,eval_metric='logloss')),
             ('GBDT',GradientBoostingClassifier(n_estimators=150,max_depth=5,random_state=42))]:
    m.fit(X_tr,y_tr); cv=cross_val_score(m,X_tr,y_tr,cv=5,scoring='accuracy')
    yp=m.predict(X_te); rp=classification_report(y_te,yp,output_dict=True)
    md_r[nm]={'cv':round(cv.mean(),4),'acc':round(m.score(X_te,y_te),4),'p':round(rp['1']['precision'],4),'r':round(rp['1']['recall'],4),'f1':round(rp['1']['f1-score'],4)}
    print(f"{nm}: CV={cv.mean():.4f}, F1={rp['1']['f1-score']:.4f}")
pd.DataFrame(md_r).T

In [ ]:
# 图14: 模型对比
fig,ax=plt.subplots(figsize=(10,6))
names=['LR','RF','XGB','GBDT']; fn=['Logistic\nRegression','Random\nForest','XGBoost','Gradient\nBoosting']
ms=['acc','p','r','f1']; ml=['准确率','精确率','召回率','F1分数']
x=np.arange(len(names)); w=0.18
for i,(metric,label) in enumerate(zip(ms,ml)):
    vals=[md_r[n][metric] for n in names]
    bars=ax.bar(x+i*w,vals,w,label=label,color=COLORS[i],alpha=0.85,edgecolor='white')
    for b,v in zip(bars,vals):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.005,f'{v:.3f}',ha='center',va='bottom',fontsize=6.5,fontweight='bold')
ax.set_xlabel('分类模型',fontproperties=FP); ax.set_ylabel('得分',fontproperties=FP)
ax.set_title('文章热度预测模型性能对比',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.set_xticks(x+w*1.5); ax.set_xticklabels(fn,fontsize=9)
ax.legend(prop=FP_LEGEND); ax.set_ylim(0,1.1)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 特征重要性+混淆矩阵
best_model=RandomForestClassifier(n_estimators=200,max_depth=15,random_state=42,n_jobs=-1)
best_model.fit(X_tr,y_tr); imp=best_model.feature_importances_
fnames=['点赞数','评论数','收藏数']+top_tags+[f'tfidf_{i}' for i in range(100)]
fidf=pd.DataFrame({'f':fnames[:len(imp)],'v':imp}).sort_values('v',ascending=False).head(20)
fig,ax=plt.subplots(figsize=(10,7))
fs=fidf.sort_values('v',ascending=True)
ax.barh(fs['f'],fs['v'],color=COLORS[0],alpha=0.85,edgecolor='white')
ax.set_xlabel('特征重要性',fontproperties=FP)
ax.set_title('Random Forest特征重要性Top20',fontproperties=FP_TITLE,weight='bold',pad=12)
for l in ax.get_yticklabels(): l.set_fontproperties(FP)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
# 混淆矩阵
yp_best=best_model.predict(X_te); cm=confusion_matrix(y_te,yp_best)
fig,ax=plt.subplots(figsize=(6,5))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=ax,xticklabels=['非热门','热门'],yticklabels=['非热门','热门'])
ax.set_xlabel('预测标签',fontproperties=FP); ax.set_ylabel('真实标签',fontproperties=FP)
ax.set_title('Random Forest混淆矩阵',fontproperties=FP_TITLE,weight='bold',pad=12)
for l in ax.get_xticklabels(): l.set_fontproperties(FP)
for l in ax.get_yticklabels(): l.set_fontproperties(FP)
plt.tight_layout(); plt.show()

## 9. 技术态度倾向分析

In [ ]:
positive_words=set(['推荐','建议','高效','优化','提升','改进','创新','突破','领先','先进','优秀','强大','灵活',
    '简洁','优雅','完美','显著','有效','快速','稳定','可靠','实用','便捷','智能','精准','卓越','前沿','全新',
    '最佳','最优','高性能','低成本','高精度','高效率','可扩展','鲁棒','深入浅出','通俗易懂','干货','实战',
    '保姆级','一站式','全面','详尽','深入','透彻','清晰','系统化','体系化','完整','详解','全解',
    '赋能','驱动','加速','助力','降本','增效','落地','闭环','利器','神器','宝藏','必看','收藏','强烈推荐'])
cautious_words=set(['问题','困难','不足','局限','挑战','缺点','缺陷','瓶颈','障碍','痛点','复杂','难以','不易',
    '失败','错误','风险','代价','负担','冗余','浪费','受限','约束','妥协','权衡','折中','取舍','矛盾','冲突',
    '过拟合','欠拟合','收敛慢','计算量大','内存不足','精度低','调参','繁琐','耗时','不透明','黑箱','不可解释',
    '待解决','仍需','尚未','有待','亟需','亟待','噪声','偏差','不稳定','不一致','缺失','异常'])
depth_words=set(['原理','推导','公式','证明','理论','本质','底层','源码','内核','架构','设计','实现','代码',
    '实战','案例','实验','对比','消融','基准','benchmark','SOTA','baseline','复现','手把手','从零开始','详细','逐步'])
print(f"词典: 积极={len(positive_words)}, 审慎={len(cautious_words)}, 深度={len(depth_words)}")

In [ ]:
def count_kw(text,ws):
    if not text or pd.isna(text): return 0
    t=str(text).lower()
    return sum(t.count(w.lower()) for w in ws)
df['pos_cnt']=df['description'].apply(lambda x:count_kw(x,positive_words))
df['cau_cnt']=df['description'].apply(lambda x:count_kw(x,cautious_words))
df['dep_cnt']=df['description'].apply(lambda x:count_kw(x,depth_words))
df['att_ratio']=df['pos_cnt']/(df['pos_cnt']+df['cau_cnt']+1)
print(f"积极词: mean={df['pos_cnt'].mean():.2f}, median={df['pos_cnt'].median():.1f}")
print(f"审慎词: mean={df['cau_cnt'].mean():.2f}, median={df['cau_cnt'].median():.1f}")
print(f"态度比: mean={df['att_ratio'].mean():.3f}, median={df['att_ratio'].median():.3f}")

In [ ]:
# 图17: 三类词频分布
fig,axes=plt.subplots(1,3,figsize=(15,5))
for ax,(col,label,color) in zip(axes,[('pos_cnt','积极推广词频',COLORS[2]),('cau_cnt','审慎问题词频',COLORS[1]),('dep_cnt','技术深度词频',COLORS[0])]):
    data=df[col]; p99=data.quantile(0.99); clipped=data[data<=p99]
    ax.hist(clipped,bins=40,color=color,alpha=0.85,edgecolor='white',linewidth=0.3)
    med=data.median(); ax.axvline(med,color='red',linestyle='--',linewidth=1.5,label=f'中位数={med:.1f}')
    ax.set_xlabel(label,fontproperties=FP); ax.set_ylabel('文章数量',fontproperties=FP)
    ax.set_title(f'{label}分布',fontproperties=FP_TITLE,weight='bold')
    for l in ax.get_xticklabels(): l.set_fontproperties(FP_SMALL)
    for l in ax.get_yticklabels(): l.set_fontproperties(FP_SMALL)
    ax.legend(prop=FP_LEGEND)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图18: 各领域态度词频对比
kw_groups={}
for _,row in df.iterrows():
    sk=str(row.get('search_keyword',''))
    for kw in sk.split(','):
        kw=kw.strip()
        if kw and kw!='nan':
            if kw not in kw_groups: kw_groups[kw]={'pos':[],'cau':[],'ratio':[]}
            kw_groups[kw]['pos'].append(row['pos_cnt']); kw_groups[kw]['cau'].append(row['cau_cnt']); kw_groups[kw]['ratio'].append(row['att_ratio'])
kw_stats=[{'kw':k,'pm':np.mean(d['pos']),'cm':np.mean(d['cau']),'rm':np.mean(d['ratio']),'n':len(d['pos'])} for k,d in kw_groups.items() if len(d['pos'])>=20]
ksdf=pd.DataFrame(kw_stats).sort_values('rm',ascending=True)
fig,ax=plt.subplots(figsize=(10,6))
y=np.arange(len(ksdf)); w=0.35
ax.barh(y-w/2,ksdf['pm'],w,color=COLORS[2],alpha=0.85,label='积极推广词均频',edgecolor='white')
ax.barh(y+w/2,ksdf['cm'],w,color=COLORS[1],alpha=0.85,label='审慎问题词均频',edgecolor='white')
ax.set_yticks(y); ax.set_yticklabels(ksdf['kw'],fontproperties=FP)
ax.set_xlabel('平均词频',fontproperties=FP)
ax.set_title('各技术领域技术态度词频对比',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.legend(prop=FP_LEGEND,loc='lower right')
for i,(_,r) in enumerate(ksdf.iterrows()):
    ax.text(max(r['pm'],r['cm'])+0.05,i,f'态度比={r["rm"]:.3f}',va='center',fontsize=8,color='gray')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图19: 年度态度趋势
yearly=df.groupby('publish_year').agg(pos=('pos_cnt','mean'),cau=('cau_cnt','mean'),ratio=('att_ratio','mean')).reset_index()
fig,ax1=plt.subplots(figsize=(10,5))
ax1.plot(yearly['publish_year'],yearly['pos'],'o-',color=COLORS[2],linewidth=2,markersize=7,label='积极词均频')
ax1.plot(yearly['publish_year'],yearly['cau'],'s-',color=COLORS[1],linewidth=2,markersize=7,label='审慎词均频')
ax1.set_xlabel('年份',fontproperties=FP); ax1.set_ylabel('平均词频',fontproperties=FP)
ax1.set_xticks(yearly['publish_year'])
for l in ax1.get_xticklabels(): l.set_fontproperties(FP_SMALL)
for l in ax1.get_yticklabels(): l.set_fontproperties(FP_SMALL)
ax2=ax1.twinx()
ax2.plot(yearly['publish_year'],yearly['ratio'],'^-',color=COLORS[3],linewidth=2,markersize=7,label='态度倾向比')
ax2.set_ylabel('态度倾向比',fontproperties=FP); ax2.set_ylim(0,1)
for l in ax2.get_yticklabels(): l.set_fontproperties(FP_SMALL)
l1,la1=ax1.get_legend_handles_labels(); l2,la2=ax2.get_legend_handles_labels()
ax1.legend(l1+l2,la1+la2,prop=FP_LEGEND,loc='upper left')
ax1.set_title('年度技术态度倾向变化趋势(2015-2026)',fontproperties=FP_TITLE,weight='bold',pad=12)
ax1.spines['top'].set_visible(False); ax2.spines['top'].set_visible(False)
plt.tight_layout(); plt.show()

## 10. 异常检测（Isolation Forest / LOF）

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
features=df[['views','likes','comments','collections','att_ratio']].copy()
for c in ['views','likes','comments','collections']: features[f'log_{c}']=np.log1p(features[c])
X_anom=features[['log_views','log_likes','log_comments','log_collections','att_ratio']].values
scaler=StandardScaler(); X_sc=scaler.fit_transform(X_anom)
iso=IsolationForest(n_estimators=200,contamination=0.05,random_state=42)
df['iso_label']=iso.fit_predict(X_sc); df['iso_score']=iso.decision_function(X_sc)
n_iso=(df['iso_label']==-1).sum(); print(f"IF异常: {n_iso}篇 ({n_iso/len(df)*100:.1f}%)")
lof=LocalOutlierFactor(n_neighbors=20,contamination=0.05)
df['lof_label']=lof.fit_predict(X_sc)
n_lof=(df['lof_label']==-1).sum(); print(f"LOF异常: {n_lof}篇 ({n_lof/len(df)*100:.1f}%)")
both=((df['iso_label']==-1)&(df['lof_label']==-1)).sum(); print(f"共同异常: {both}篇")

In [ ]:
# 图20: IF异常分数分布
fig,ax=plt.subplots(figsize=(10,5))
nm=df[df['iso_label']==1]['iso_score']; am=df[df['iso_label']==-1]['iso_score']
ax.hist(nm,bins=60,color=COLORS[0],alpha=0.7,label=f'正常({len(nm)}篇)',edgecolor='white',linewidth=0.3)
ax.hist(am,bins=30,color=COLORS[3],alpha=0.8,label=f'异常({len(am)}篇)',edgecolor='white',linewidth=0.3)
ax.axvline(0,color='red',linestyle='--',linewidth=1.5,label='判定阈值')
ax.set_xlabel('异常分数(越小越异常)',fontproperties=FP); ax.set_ylabel('文章数量',fontproperties=FP)
ax.set_title('Isolation Forest异常分数分布',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.legend(prop=FP_LEGEND)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图21: 异常散点图
fig,ax=plt.subplots(figsize=(10,7))
nm=df['iso_label']==1; am=df['iso_label']==-1
ax.scatter(df.loc[nm,'views'],df.loc[nm,'likes'],c=COLORS[0],alpha=0.3,s=10,label=f'正常({nm.sum()}篇)')
ax.scatter(df.loc[am,'views'],df.loc[am,'likes'],c=COLORS[3],alpha=0.8,s=30,marker='x',linewidths=1.5,label=f'异常({am.sum()}篇)')
ax.set_xlabel('浏览量',fontproperties=FP); ax.set_ylabel('点赞数',fontproperties=FP)
ax.set_title('Isolation Forest异常检测: 浏览量 vs 点赞数',fontproperties=FP_TITLE,weight='bold',pad=12)
ax.set_xscale('log'); ax.set_yscale('log')
ax.legend(prop=FP_LEGEND,loc='upper left')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# 图22: IF vs LOF对比 + 异常类型分析
adf=df[df['iso_label']==-1].copy()
adf['lvr']=adf['likes']/(adf['views']+1)
hvl=adf[(adf['views']>adf['views'].median())&(adf['lvr']<adf['lvr'].quantile(0.25))]
lvh=adf[(adf['views']<adf['views'].median())&(adf['likes']>adf['likes'].median())]
print(f"高浏览低点赞(标题党): {len(hvl)}篇")
print(f"低浏览高点赞(冷门宝藏): {len(lvh)}篇")
fig,axes=plt.subplots(1,2,figsize=(12,5))
bars=axes[0].bar(['IF','LOF','交集'],[n_iso,n_lof,both],color=[COLORS[0],COLORS[1],COLORS[2]],alpha=0.85,edgecolor='white')
for b,v in zip(bars,[n_iso,n_lof,both]):
    axes[0].text(b.get_x()+b.get_width()/2,b.get_height()+3,str(v),ha='center',fontweight='bold',fontsize=11)
axes[0].set_ylabel('异常文章数',fontproperties=FP)
axes[0].set_title('两种异常检测方法对比',fontproperties=FP_TITLE,weight='bold',pad=12)
for l in axes[0].get_xticklabels(): l.set_fontproperties(FP)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
anv=df[df['iso_label']==-1]['views']; norv=df[df['iso_label']==1]['views']
axes[1].hist(np.log1p(anv),bins=40,color=COLORS[3],alpha=0.7,label='异常',edgecolor='white')
axes[1].hist(np.log1p(norv),bins=40,color=COLORS[0],alpha=0.5,label='正常',edgecolor='white')
axes[1].set_xlabel('log(1+浏览量)',fontproperties=FP); axes[1].set_ylabel('文章数量',fontproperties=FP)
axes[1].set_title('异常vs正常文章浏览量分布',fontproperties=FP_TITLE,weight='bold',pad=12)
axes[1].legend(prop=FP_LEGEND)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 分析完成

本报告涵盖10种数据挖掘方法，共生成22张可视化图表和5张数据表格。